# Graph OLAP — Quick Start

This notebook shows how to:
1. Connect to the Graph OLAP platform
2. Load the **IPL Cricket** sample graph
3. Load the **Datacenter Infrastructure** sample graph
4. Run Cypher queries
5. Visualize results

---

## Setup

In [ ]:
from graph_olap_sdk import GraphOLAPClient

# Connect to the control plane
client = GraphOLAPClient("http://graph-olap-control-plane:8080")
print(f"Connected — {len(client.instances.list())} instances running")

## 1. IPL Cricket Graph

A graph of IPL T20 cricket with **Teams**, **Players**, and **Games** connected by PLAYS_FOR, PLAYED_IN, and WON relationships.

In [ ]:
import pandas as pd
import numpy as np
import requests, os

np.random.seed(42)

# --- Generate data ---
teams = pd.DataFrame({
    "team_id": range(1, 9),
    "name": ["Mumbai Indians", "Chennai Super Kings", "Royal Challengers", "Kolkata Knight Riders",
             "Delhi Capitals", "Rajasthan Royals", "Sunrisers Hyderabad", "Punjab Kings"],
    "city": ["Mumbai", "Chennai", "Bangalore", "Kolkata", "Delhi", "Jaipur", "Hyderabad", "Mohali"],
    "titles": [5, 5, 0, 2, 0, 1, 1, 0],
})

players = pd.DataFrame({
    "player_id": range(1, 16),
    "name": ["Rohit Sharma", "Virat Kohli", "MS Dhoni", "KL Rahul", "Jasprit Bumrah",
             "Ravindra Jadeja", "Hardik Pandya", "Rishabh Pant", "Shubman Gill", "Yuzvendra Chahal",
             "Suryakumar Yadav", "Mohammed Shami", "Jos Buttler", "Rashid Khan", "David Warner"],
    "role": ["Batsman"]*4 + ["Bowler", "All-rounder", "All-rounder", "Wicketkeeper", "Batsman", "Bowler",
             "Batsman", "Bowler", "Batsman", "Bowler", "Batsman"],
})

games = pd.DataFrame({
    "match_id": range(1, 11),
    "season": [2023]*5 + [2024]*5,
    "venue": ["Wankhede", "Chepauk", "Chinnaswamy", "Eden Gardens", "Kotla"]*2,
    "winner_team_id": [1, 2, 4, 1, 3, 2, 5, 1, 6, 2],
})

plays_for = pd.DataFrame({"player_id": range(1, 16), "team_id": [1,3,2,8,1,2,1,5,4,3,1,4,6,7,5]})
played_in_rows = []
for mid in range(1, 11):
    for pid in np.random.choice(range(1, 16), size=6, replace=False):
        played_in_rows.append({"player_id": int(pid), "match_id": mid, "runs": int(np.random.randint(0, 100)), "wickets": int(np.random.randint(0, 4))})
played_in = pd.DataFrame(played_in_rows)
won = pd.DataFrame({"team_id": games["winner_team_id"], "match_id": range(1, 11)})

print(f"Generated: {len(teams)} teams, {len(players)} players, {len(games)} games")
print(f"Edges: {len(plays_for)} plays_for, {len(played_in)} played_in, {len(won)} won")
teams.head()

In [ ]:
# --- Create mapping ---
mapping_id = client.mappings.create(
    name="IPL T20 Cricket",
    description="[via:jupyter] IPL cricket graph",
    node_definitions=[
        {"label": "Team", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "team_id", "type": "INT64"}, "properties": [{"name": "name", "type": "STRING"}, {"name": "city", "type": "STRING"}, {"name": "titles", "type": "INT64"}]},
        {"label": "Player", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "player_id", "type": "INT64"}, "properties": [{"name": "name", "type": "STRING"}, {"name": "role", "type": "STRING"}]},
        {"label": "Game", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "match_id", "type": "INT64"}, "properties": [{"name": "season", "type": "INT64"}, {"name": "venue", "type": "STRING"}, {"name": "winner_team_id", "type": "INT64"}]},
    ],
    edge_definitions=[
        {"type": "PLAYS_FOR", "from_node": "Player", "to_node": "Team", "sql": "SELECT * FROM placeholder", "from_key": "player_id", "to_key": "team_id", "properties": []},
        {"type": "PLAYED_IN", "from_node": "Player", "to_node": "Game", "sql": "SELECT * FROM placeholder", "from_key": "player_id", "to_key": "match_id", "properties": [{"name": "runs", "type": "INT64"}, {"name": "wickets", "type": "INT64"}]},
        {"type": "WON", "from_node": "Team", "to_node": "Game", "sql": "SELECT * FROM placeholder", "from_key": "team_id", "to_key": "match_id", "properties": []},
    ]
)
print(f"Mapping created: {mapping_id}")

In [ ]:
# --- Upload parquet to GCS and create instance ---
base = "/tmp/ipl-graph"
for d in ["nodes/Team", "nodes/Player", "nodes/Game", "edges/PLAYS_FOR", "edges/PLAYED_IN", "edges/WON"]:
    os.makedirs(f"{base}/{d}", exist_ok=True)

teams.to_parquet(f"{base}/nodes/Team/data.parquet", index=False)
players.to_parquet(f"{base}/nodes/Player/data.parquet", index=False)
games.to_parquet(f"{base}/nodes/Game/data.parquet", index=False)
plays_for.to_parquet(f"{base}/edges/PLAYS_FOR/data.parquet", index=False)
played_in.to_parquet(f"{base}/edges/PLAYED_IN/data.parquet", index=False)
won.to_parquet(f"{base}/edges/WON/data.parquet", index=False)

# Create instance (snapshot auto-created and marked ready because SQL has 'placeholder')
inst = client.instances.create_and_wait(mapping_id, name="IPL T20 Demo", description="[via:jupyter]")
print(f"Instance running: {inst['id']}")

In [ ]:
# --- Upload data to GCS ---
snapshot_id = inst["snapshot_id"]
GCS = "http://fake-gcs-local:4443"
BUCKET = "graph-olap-local-dev"
OWNER = "demo@example.com"
PREFIX = f"{OWNER}/{mapping_id}/v1/{snapshot_id}"

# Ensure bucket exists
_r = requests.post(f"{GCS}/storage/v1/b", json={"name": BUCKET})

files = [
    (f"{base}/nodes/Team/data.parquet",     f"{PREFIX}/nodes/Team/data.parquet"),
    (f"{base}/nodes/Player/data.parquet",   f"{PREFIX}/nodes/Player/data.parquet"),
    (f"{base}/nodes/Game/data.parquet",     f"{PREFIX}/nodes/Game/data.parquet"),
    (f"{base}/edges/PLAYS_FOR/data.parquet", f"{PREFIX}/edges/PLAYS_FOR/data.parquet"),
    (f"{base}/edges/PLAYED_IN/data.parquet", f"{PREFIX}/edges/PLAYED_IN/data.parquet"),
    (f"{base}/edges/WON/data.parquet",      f"{PREFIX}/edges/WON/data.parquet"),
]

for local_path, remote_name in files:
    url = f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote_name}"
    with open(local_path, "rb") as f:
        r = requests.post(url, data=f.read(), headers={"Content-Type": "application/octet-stream"})
    print(f"  {'OK' if r.status_code == 200 else 'FAIL'} {remote_name.split('/')[-2]}/{remote_name.split('/')[-1]}")

print(f"\nData uploaded to gs://{BUCKET}/{PREFIX}/")

## 2. Query the IPL Graph

In [ ]:
wrapper_url = f"http://{inst['pod_name']}:8000"

def query(cypher):
    r = requests.post(f"{wrapper_url}/query", json={"query": cypher})
    data = r.json()
    return pd.DataFrame(data["rows"], columns=data["columns"])

# Node counts
query("MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count ORDER BY count DESC")

In [ ]:
# Which team has won the most matches?
query("""
MATCH (t:Team)-[:WON]->(g:Game)
RETURN t.name AS team, count(g) AS wins
ORDER BY wins DESC
""")

In [ ]:
# Players who played in the most games
query("""
MATCH (p:Player)-[:PLAYED_IN]->(g:Game)
RETURN p.name AS player, p.role AS role, count(g) AS games_played
ORDER BY games_played DESC LIMIT 10
""")

In [ ]:
# Top scorers
query("""
MATCH (p:Player)-[pi:PLAYED_IN]->(g:Game)
RETURN p.name AS player, sum(pi.runs) AS total_runs, sum(pi.wickets) AS total_wickets
ORDER BY total_runs DESC LIMIT 10
""")

---

## 3. Datacenter Infrastructure Graph

A graph of **Datacenters**, **Racks**, **Servers**, and **Applications** with LOCATED_IN, INSTALLED_IN, and HOSTED_ON relationships.

In [ ]:
import random
random.seed(42)

# --- Generate datacenter data ---
dcs = pd.DataFrame({
    "dc_id": [f"dc-{i:02d}" for i in range(1, 6)],
    "name": ["London-1", "Frankfurt-1", "Virginia-1", "Mumbai-1", "Singapore-1"],
    "location": ["London, UK", "Frankfurt, DE", "Ashburn, US", "Mumbai, IN", "Singapore, SG"],
    "tier": ["Tier 4", "Tier 3", "Tier 4", "Tier 3", "Tier 3"],
})

rack_rows = []
rack_num = 1
for dc in dcs["dc_id"]:
    for row in ["A", "B", "C"]:
        for pos in range(1, 5):
            rack_rows.append({"rack_id": f"rack-{rack_num:03d}", "dc_id": dc, "row": row, "position": pos})
            rack_num += 1
racks = pd.DataFrame(rack_rows)

srv_rows = []
srv_num = 1
for rack_id in racks["rack_id"]:
    for _ in range(random.randint(3, 5)):
        srv_rows.append({
            "server_id": f"srv-{srv_num:04d}", "rack_id": rack_id,
            "hostname": f"node-{srv_num:04d}.internal",
            "cpu_cores": random.choice([16, 32, 64, 128]),
            "ram_gb": random.choice([64, 128, 256, 512]),
            "status": random.choice(["running"]*4 + ["maintenance", "offline"]),
        })
        srv_num += 1
servers = pd.DataFrame(srv_rows)

apps = pd.DataFrame({
    "app_id": [f"app-{i:02d}" for i in range(1, 11)],
    "name": ["Auth Service", "Payment Gateway", "User API", "Analytics Pipeline", "ML Training",
             "Dashboard UI", "Notification Service", "Search Index", "CI/CD Runner", "Monitoring Stack"],
    "team": ["Platform", "Payments", "Platform", "Data", "AI", "Frontend", "Platform", "Data", "DevOps", "SRE"],
    "criticality": ["Critical", "Critical", "High", "Medium", "Low", "High", "Medium", "High", "Medium", "Critical"],
})

# Edges
located_in = racks[["rack_id", "dc_id"]].copy()
installed_in = servers[["server_id", "rack_id"]].copy()
hosted_rows = []
for app_id in apps["app_id"]:
    for srv in random.sample(list(servers["server_id"]), random.randint(2, 5)):
        hosted_rows.append({"app_id": app_id, "server_id": srv})
hosted_on = pd.DataFrame(hosted_rows)

print(f"Datacenters: {len(dcs)}, Racks: {len(racks)}, Servers: {len(servers)}, Apps: {len(apps)}")
print(f"Edges: {len(located_in)} located_in, {len(installed_in)} installed_in, {len(hosted_on)} hosted_on")

In [ ]:
# --- Create mapping + instance ---
dc_mapping_id = client.mappings.create(
    name="Datacenter Infrastructure",
    description="[via:jupyter] DC infra graph",
    node_definitions=[
        {"label": "Datacenter", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "dc_id", "type": "STRING"}, "properties": [{"name": "name", "type": "STRING"}, {"name": "location", "type": "STRING"}, {"name": "tier", "type": "STRING"}]},
        {"label": "Rack", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "rack_id", "type": "STRING"}, "properties": [{"name": "dc_id", "type": "STRING"}, {"name": "row", "type": "STRING"}, {"name": "position", "type": "INT64"}]},
        {"label": "Server", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "server_id", "type": "STRING"}, "properties": [{"name": "rack_id", "type": "STRING"}, {"name": "hostname", "type": "STRING"}, {"name": "cpu_cores", "type": "INT64"}, {"name": "ram_gb", "type": "INT64"}, {"name": "status", "type": "STRING"}]},
        {"label": "Application", "sql": "SELECT * FROM placeholder", "primary_key": {"name": "app_id", "type": "STRING"}, "properties": [{"name": "name", "type": "STRING"}, {"name": "team", "type": "STRING"}, {"name": "criticality", "type": "STRING"}]},
    ],
    edge_definitions=[
        {"type": "LOCATED_IN", "from_node": "Rack", "to_node": "Datacenter", "sql": "SELECT * FROM placeholder", "from_key": "rack_id", "to_key": "dc_id", "properties": []},
        {"type": "INSTALLED_IN", "from_node": "Server", "to_node": "Rack", "sql": "SELECT * FROM placeholder", "from_key": "server_id", "to_key": "rack_id", "properties": []},
        {"type": "HOSTED_ON", "from_node": "Application", "to_node": "Server", "sql": "SELECT * FROM placeholder", "from_key": "app_id", "to_key": "server_id", "properties": []},
    ]
)

dc_inst = client.instances.create_and_wait(dc_mapping_id, name="Datacenter Infra", description="[via:jupyter]")
print(f"Datacenter instance running: {dc_inst['id']}")

In [ ]:
# --- Upload datacenter data to GCS ---
dc_base = "/tmp/dc-graph"
for d in ["nodes/Datacenter", "nodes/Rack", "nodes/Server", "nodes/Application",
          "edges/LOCATED_IN", "edges/INSTALLED_IN", "edges/HOSTED_ON"]:
    os.makedirs(f"{dc_base}/{d}", exist_ok=True)

dcs.to_parquet(f"{dc_base}/nodes/Datacenter/data.parquet", index=False)
racks.to_parquet(f"{dc_base}/nodes/Rack/data.parquet", index=False)
servers.to_parquet(f"{dc_base}/nodes/Server/data.parquet", index=False)
apps.to_parquet(f"{dc_base}/nodes/Application/data.parquet", index=False)
located_in.to_parquet(f"{dc_base}/edges/LOCATED_IN/data.parquet", index=False)
installed_in.to_parquet(f"{dc_base}/edges/INSTALLED_IN/data.parquet", index=False)
hosted_on.to_parquet(f"{dc_base}/edges/HOSTED_ON/data.parquet", index=False)

dc_snapshot_id = dc_inst["snapshot_id"]
DC_PREFIX = f"{OWNER}/{dc_mapping_id}/v1/{dc_snapshot_id}"

for local, remote in [
    (f"{dc_base}/nodes/Datacenter/data.parquet",  f"{DC_PREFIX}/nodes/Datacenter/data.parquet"),
    (f"{dc_base}/nodes/Rack/data.parquet",        f"{DC_PREFIX}/nodes/Rack/data.parquet"),
    (f"{dc_base}/nodes/Server/data.parquet",      f"{DC_PREFIX}/nodes/Server/data.parquet"),
    (f"{dc_base}/nodes/Application/data.parquet", f"{DC_PREFIX}/nodes/Application/data.parquet"),
    (f"{dc_base}/edges/LOCATED_IN/data.parquet",  f"{DC_PREFIX}/edges/LOCATED_IN/data.parquet"),
    (f"{dc_base}/edges/INSTALLED_IN/data.parquet",f"{DC_PREFIX}/edges/INSTALLED_IN/data.parquet"),
    (f"{dc_base}/edges/HOSTED_ON/data.parquet",   f"{DC_PREFIX}/edges/HOSTED_ON/data.parquet"),
]:
    url = f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote}"
    with open(local, "rb") as f:
        requests.post(url, data=f.read(), headers={"Content-Type": "application/octet-stream"})

print("Datacenter data uploaded!")

## 4. Query the Datacenter Graph

In [ ]:
dc_wrapper = f"http://{dc_inst['pod_name']}:8000"

def dc_query(cypher):
    r = requests.post(f"{dc_wrapper}/query", json={"query": cypher})
    data = r.json()
    return pd.DataFrame(data["rows"], columns=data["columns"])

# Node counts
dc_query("MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count ORDER BY count DESC")

In [ ]:
# How many servers per datacenter?
dc_query("""
MATCH (s:Server)-[:INSTALLED_IN]->(r:Rack)-[:LOCATED_IN]->(dc:Datacenter)
RETURN dc.name AS datacenter, dc.location AS location, count(s) AS servers
ORDER BY servers DESC
""")

In [ ]:
# Which apps are critical and how many servers do they run on?
dc_query("""
MATCH (a:Application)-[:HOSTED_ON]->(s:Server)
WHERE a.criticality = 'Critical'
RETURN a.name AS app, a.team AS team, count(s) AS server_count
ORDER BY server_count DESC
""")

In [ ]:
# Find servers that are offline
dc_query("""
MATCH (s:Server)-[:INSTALLED_IN]->(r:Rack)-[:LOCATED_IN]->(dc:Datacenter)
WHERE s.status = 'offline'
RETURN s.hostname AS server, dc.name AS datacenter, r.rack_id AS rack
""")

In [ ]:
# Impact analysis: if a rack goes down, which apps are affected?
dc_query("""
MATCH (a:Application)-[:HOSTED_ON]->(s:Server)-[:INSTALLED_IN]->(r:Rack)
WHERE r.rack_id = 'rack-001'
RETURN DISTINCT a.name AS affected_app, a.criticality AS criticality, count(s) AS servers_on_rack
ORDER BY criticality
""")

---

## 5. Cleanup

Run this cell to terminate instances when done.

In [ ]:
# Uncomment to clean up:
# client.instances.terminate(inst['id'])
# client.instances.terminate(dc_inst['id'])
# print("Instances terminated")